- Spark Master UI → http://localhost:8080
- Spark Worker UI → http://localhost:8081 (haven't tried it yet)

In [1]:
!pyspark --version        # or alternatively (same output): spark-submit --version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/05 15:59:20 WARN Utils: Your hostname, sohang-VivoBook-ASUS-Laptop-X510UFO, resolves to a loopback address: 127.0.1.1; using 192.168.1.31 instead (on interface wlp2s0)
26/07/05 15:59:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 4.1.1
      /_/
                        
Using Scala version 2.13.17, OpenJDK 64-Bit Server VM, 17.0.19
Branch HEAD
Compiled by user runner on 2026-01-02T11:55:02Z
Revision c0690c763bafabd08e7079d1137fa0a769a05bae
Url https://github.com/apache/spark
Type --help for more information.


In [2]:
!~/kafka_2.13-4.3.0/bin/kafka-topics.sh --version

4.3.0


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType
from pyspark.sql.window import Window


In [ ]:
session = (
    SparkSession.builder.appName("SensorStreamingPipeline")
    .config("spark.sql.streaming.schemaInference", "true")
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
    .config(
        # FINALLY THIS JAR WORKS!!!
        "spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.2"
    )
    .getOrCreate()
)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/05 15:59:25 WARN Utils: Your hostname, sohang-VivoBook-ASUS-Laptop-X510UFO, resolves to a loopback address: 127.0.1.1; using 192.168.1.31 instead (on interface wlp2s0)
26/07/05 15:59:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/sohang/.local/bin/miniconda3/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/sohang/.ivy2.5.2/cache
The jars for the packages stored in: /home/sohang/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-73374aba-42a9-4e16-9d85-b957befe2c6d;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.1.2 in central
	found org.apache.kafka#kaf

In [ ]:
schema = StructType(
    [
        StructField("sensor_id", StringType(), False),
        StructField("temperature", DoubleType(), True),
        StructField("timestamp", StringType(), False),
        StructField("status", StringType(), False),
    ]
)

In [17]:
kafka_site = "localhost:9092"
topic = "sensor_DA25M622"
raw_df = (
    session.readStream.format("kafka")
    .option("kafka.bootstrap.servers", kafka_site)
    .option("subscribe", topic)
    .option("startingOffsets", "earliest")
    .load()
)


In [21]:
# parsed df according to schema
parsed_json_df = raw_df.select(
    F.from_json(F.col("value").cast(F.StringType()), schema).alias("data")
).select("data.*")
parsed_json_df.printSchema()

root
 |-- sensor_id: string (nullable = true)
 |-- temperature: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- status: string (nullable = true)



In [37]:
# timestamps: parse from strings
parsed_timestamps_df = (
    # .cast() would raise error on invalid string
    parsed_json_df.withColumn("timestamp", F.col("timestamp").try_cast("timestamp"))  # .try_cast() returns null on invalid string so we filter it out
    .filter(F.col("timestamp").isNotNull())  
)

In [ ]:
# missing_temp_df = parsed_json_df.filter(F.col("temperature").isNull())

# SKIPPING AS CAN'T GET IT TO WORK: Replace missing temperature values using the average temperature of the same sensor
# over the previous 5-minute window, OR report and drop records if insufficient history
# exists. (getting empty data with following code) Instead I'll just use a random fill value 25 temperature
df = (
    parsed_timestamps_df.withColumn(
        "temperature",
        F.when(F.col("temperature").isNull(), 25.0).otherwise(F.col("temperature")),
    )
    .filter(F.col("temperature").isNotNull())
)


# watermark_delay = "1 seconds"

# # 1. Main stream gets its watermark
# stream_df = parsed_timestamps_df.withWatermark("timestamp", watermark_delay)

# # 2. History stream: aggregate and explicitly apply a watermark to the output
# history_df = (
#     stream_df
#     .filter(F.col("temperature").isNotNull())
#     .groupBy(
#         F.col("sensor_id").alias("hist_sensor_id"),
#         F.window("timestamp", "5 minutes", "1 minute").alias("time_window")
#     )
#     .agg(F.avg("temperature").alias("avg_temp_5m"))
#     .select(
#         "hist_sensor_id",
#         F.col("time_window.start").alias("win_start"),
#         F.col("time_window.end").alias("win_end"),
#         "avg_temp_5m"
#     )
#     # CRITICAL: Re-apply watermark on the generated time column for the right side
#     .withWatermark("win_start", watermark_delay)
# )

# # 3. Join matching both join keys (sensor_id) AND a direct time range condition
# joined_df = stream_df.join(
#     other=history_df,
#     on=F.expr("""
#         sensor_id = hist_sensor_id AND
#         timestamp >= win_start AND
#         timestamp < win_end
#     """),
#     how="left"
# )

# # 4. Coalesce and drop rows with empty history
# df = (
#     joined_df
#     .withColumn("temperature", F.coalesce(F.col("temperature"), F.col("avg_temp_5m")))
#     .filter(F.col("temperature").isNotNull())
#     .select("sensor_id", "temperature", "timestamp", "status")
# )

In [57]:
df = df.dropDuplicates(["sensor_id", "timestamp"])

In [59]:
df = df.filter((F.col("temperature") > -20) & (F.col("temperature") < 100))

In [63]:
df = df.withColumn("hour", F.hour(F.col("timestamp")))
df = df.withColumn("day_of_week", F.dayofweek(F.col("timestamp")))
df = df.withColumn("is_weekend", F.when(F.col("day_of_week").isin([1, 7]), True).otherwise(False))

In [ ]:
# spark dataframes
watermarked = df.withWatermark("timestamp", "1 second")
avg_temp_per_sensor = watermarked.groupBy("sensor_id").agg(
    F.avg("temperature").alias("avg_temperature")
)
max_temp_per_sensor = watermarked.groupBy("sensor_id").agg(
    F.max("temperature").alias("max_temperature")
)
active_sensors = (
    watermarked.withColumn("window_time", F.window(F.col("timestamp"), "5 minutes"))
    .groupBy("window_time")
    .agg(F.count(F.col("sensor_id")).alias("active_sensor_count"))
)
status_distribution = watermarked.groupBy("status").agg(F.count("*").alias("count"))


In [75]:
# avg, tumbling window, 5 minutes, watermark also 5 minutes
windowed_avg = (
    df.withWatermark("timestamp", "5 minutes")
    .groupBy(
        F.window(F.col("timestamp"), "5 minutes"), F.col("sensor_id")
    )
    .agg(F.avg("temperature").alias("window_avg_temperature"))
)

In [ ]:
def query_show(df):
    query = (
        df.writeStream
        .format("console")
        #.outputMode("append")        
        .outputMode("update")      # show output as soon as it arrives, for groupby since with append, error: Invalid streaming output mode: append. This output mode is not supported for streaming aggregations without watermark on streaming DataFrames/DataSets. 
        #.trigger(processingTime="2 seconds") 
        .trigger(availableNow=True)    # process only data currently available (2000 records in kafka), then close stream
        .start()
    )

q = query_show(avg_temp_per_sensor)

26/07/05 18:37:55 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-98a51ef2-c803-4f2b-b75f-a770ae3b388f. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/05 18:37:55 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/07/05 18:37:55 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+------------------+
|sensor_id|   avg_temperature|
+---------+------------------+
| sensor_7|  25.5604347826087|
| sensor_8|26.243260869565212|
| sensor_1|24.150816326530613|
| sensor_4|26.257659574468082|
| sensor_9|24.579347826086963|
| sensor_6|24.305333333333333|
|sensor_10|25.340999999999998|
| sensor_2|25.818510638297862|
| sensor_3| 25.27295454545454|
| sensor_5| 24.24519230769232|
+---------+------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------+---------------+
|sensor_id|avg_temperature|
+---------+---------------+
+---------+---------------+



In [ ]:
q = query_show(max_temp_per_sensor)

26/07/05 18:40:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-93ebb902-d727-4243-89e5-3e3571c5ccf5. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/05 18:40:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/07/05 18:40:01 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+---------------+
|sensor_id|max_temperature|
+---------+---------------+
| sensor_7|          34.59|
| sensor_8|          34.75|
| sensor_1|          33.71|
| sensor_4|           34.9|
| sensor_9|           34.7|
| sensor_6|          34.71|
|sensor_10|          34.82|
| sensor_2|          34.74|
| sensor_3|          32.49|
| sensor_5|          34.86|
+---------+---------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+---------+---------------+
|sensor_id|max_temperature|
+---------+---------------+
+---------+---------------+



In [73]:
q = query_show(active_sensors)

26/07/05 18:40:43 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-022e6771-db40-431a-adad-9a9268060a4d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/05 18:40:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/07/05 18:40:43 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+-------------------+
|         window_time|active_sensor_count|
+--------------------+-------------------+
|{2026-07-05 09:50...|                  2|
|{2026-07-05 10:00...|                 29|
|{2026-07-05 09:55...|                 34|
|{2026-07-05 10:10...|                270|
|{2026-07-05 10:05...|                137|
+--------------------+-------------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-------------------+
|window_time|active_sensor_count|
+-----------+-------------------+
+-----------+-------------------+



In [74]:
q = query_show(status_distribution)

26/07/05 18:41:23 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c626e4db-c1f7-4218-949e-e041a49105b8. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/05 18:41:23 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/07/05 18:41:23 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+-----+
|     status|count|
+-----------+-----+
|maintenance|   20|
|     active|  343|
|      error|   50|
|       idle|   59|
+-----------+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+------+-----+
|status|count|
+------+-----+
+------+-----+



In [76]:
q = query_show(windowed_avg)

26/07/05 18:48:58 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-b949d9f4-a7d7-49ac-aeb4-d5ca93626326. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/05 18:48:58 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/07/05 18:48:58 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+---------+----------------------+
|              window|sensor_id|window_avg_temperature|
+--------------------+---------+----------------------+
|{2026-07-05 10:00...| sensor_1|               26.4075|
|{2026-07-05 10:10...| sensor_2|    25.558846153846158|
|{2026-07-05 10:05...|sensor_10|    24.686666666666667|
|{2026-07-05 10:00...| sensor_5|    23.941666666666666|
|{2026-07-05 09:55...| sensor_8|                29.355|
|{2026-07-05 10:05...| sensor_8|    28.025000000000006|
|{2026-07-05 10:10...| sensor_5|     23.85592592592593|
|{2026-07-05 09:55...| sensor_1|    21.653333333333336|
|{2026-07-05 09:55...| sensor_3|    27.366666666666664|
|{2026-07-05 10:05...| sensor_4|    25.874285714285712|
|{2026-07-05 09:55...| sensor_7|     26.19333333333333|
|{2026-07-05 10:00...| sensor_7|                 33.54|
|{2026-07-05 10:10...| sensor_8|     25.11384615384615|
|{2026-

-------------------------------------------
Batch: 1
-------------------------------------------
+------+---------+----------------------+
|window|sensor_id|window_avg_temperature|
+------+---------+----------------------+
+------+---------+----------------------+



In [77]:
q = query_show(df)

26/07/05 18:55:24 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-299a7c7c-262a-49fe-8d0c-2689f234311e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/07/05 18:55:24 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/07/05 18:55:24 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.


-------------------------------------------
Batch: 0
-------------------------------------------
+---------+-----------+-------------------+------+----+-----------+----------+
|sensor_id|temperature|          timestamp|status|hour|day_of_week|is_weekend|
+---------+-----------+-------------------+------+----+-----------+----------+
| sensor_2|      31.65|2026-07-05 10:10:14|active|  10|          1|      true|
| sensor_4|      23.54|2026-07-05 10:10:21|active|  10|          1|      true|
| sensor_1|      19.91|2026-07-05 10:10:24|active|  10|          1|      true|
| sensor_2|      25.85|2026-07-05 10:09:48|active|  10|          1|      true|
| sensor_1|      16.51|2026-07-05 10:09:47|active|  10|          1|      true|
|sensor_10|      23.78|2026-07-05 10:10:24|active|  10|          1|      true|
| sensor_3|       31.9|2026-07-05 10:09:58|active|  10|          1|      true|
| sensor_7|      17.42|2026-07-05 10:10:27|active|  10|          1|      true|
| sensor_8|      31.11|2026-07-05 

In [ ]:
## IGNORE, ERRORING! trying to do last remaining part of: Inject late events and demonstrate: (i) accepted records (ii) discarded records

# # Assuming 'df' contains your 2000 records from Kafka 
# # (where max timestamp is ~10:10:27, setting watermark to 10:05:27)

# from datetime import datetime

# # Let's create two specific test records to inject manually
# late_data_data = [
#     # 1. ACCEPTED RECORD (Late, but inside the allowed window)
#     # This falls into the 10:05:00 - 10:10:00 window. 
#     # Since 10:06:00 is GREATER than our watermark (10:05:27), Spark accepts it.
#     ("sensor_1", 25.0, datetime.strptime("2026-07-05 10:06:00", "%Y-%m-%d %H:%M:%S"), "active", 10, 1, True),
    
#     # 2. DISCARDED RECORD (Too late, past the watermark threshold)
#     # This falls into the 10:00:00 - 10:05:00 window.
#     # Since 10:01:00 is LESS than our watermark (10:05:27), Spark drops it.
#     ("sensor_1", 99.0, datetime.strptime("2026-07-05 10:01:00", "%Y-%m-%d %H:%M:%S"), "active", 10, 1, True)
# ]

# # Define the matching schema
# schema = df.schema

# # Create a local DataFrame for the late injections
# late_df = session.createDataFrame(late_data_data, schema=schema)

# # Combine your original data with these specific test inputs
# demonstration_df = df.union(late_df)

# # Run your windowing logic on the combined dataset
# windowed_avg = (
#     demonstration_df.withWatermark("timestamp", "5 minutes")
#     .groupBy(F.window(F.col("timestamp"), "5 minutes"), F.col("sensor_id"))
#     .agg(F.avg("temperature").alias("window_avg_temperature"))
#     .select("window.start", "window.end", "sensor_id", "window_avg_temperature")
#     .orderBy("window.start")
# )

# # Output the results to the console
# query = (
#     windowed_avg.writeStream
#     .format("console")
#     .outputMode("update")
#     .trigger(availableNow=True)
#     .start()
# )